In [26]:
import polars as pl

movies = pl.read_csv("../data/raw/movies.csv")
ratings = pl.read_csv("../data/raw/ratings.csv")

movies = movies.with_columns(
    pl.col("genres")
    .str.split("|")
    .alias("genre_list")
)

print(movies.head())
print(ratings.head())

shape: (5, 4)
┌─────────┬─────────────────────────────┬─────────────────────────────┬────────────────────────────┐
│ movieId ┆ title                       ┆ genres                      ┆ genre_list                 │
│ ---     ┆ ---                         ┆ ---                         ┆ ---                        │
│ i64     ┆ str                         ┆ str                         ┆ list[str]                  │
╞═════════╪═════════════════════════════╪═════════════════════════════╪════════════════════════════╡
│ 1       ┆ Toy Story (1995)            ┆ Adventure|Animation|Childre ┆ ["Adventure", "Animation", │
│         ┆                             ┆ n|C…                        ┆ … "…                       │
│ 2       ┆ Jumanji (1995)              ┆ Adventure|Children|Fantasy  ┆ ["Adventure", "Children",  │
│         ┆                             ┆                             ┆ "Fan…                      │
│ 3       ┆ Grumpier Old Men (1995)     ┆ Comedy|Romance              ┆ ["Com

In [20]:
movies_rating_stats = (
    ratings
    .group_by("movieId")
    .agg([
        pl.len().alias("rating_count"),
        pl.col("rating").mean().alias("average_rating")
    ])
    .sort("rating_count", descending=True)
)

movies_with_ratings = movies.join(
    movies_rating_stats,
    on="movieId",
    how="left"
)

movies_with_ratings.select(
    ["title", "genres", "rating_count", "average_rating"]
).head(20)

title,genres,rating_count,average_rating
str,str,u32,f64
"""Toy Story (1995)""","""Adventure|Animation|Children|C…",68997,3.897438
"""Jumanji (1995)""","""Adventure|Children|Fantasy""",28904,3.275758
"""Grumpier Old Men (1995)""","""Comedy|Romance""",13134,3.139447
"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance""",2806,2.845331
"""Father of the Bride Part II (1…","""Comedy""",13154,3.059602
…,…,…,…
"""Casino (1995)""","""Crime|Drama""",22298,3.838349
"""Sense and Sensibility (1995)""","""Drama|Romance""",22251,3.945126
"""Four Rooms (1995)""","""Comedy""",6191,3.403489


In [21]:
movies = movies.with_columns(
    pl.col("movieId").cast(pl.Int64)
)

ratings = ratings.with_columns(
    pl.col("movieId").cast(pl.Int64)
)

In [27]:
def genre_similarity(genres1, genres2):
    set1 = set(genres1)
    set2 = set(genres2)

    if not set1 or not set2:
        return 0.0

    intersection = len(set1 & set2)
    union = len(set1 | set2)

    return intersection / union

# Get similar movies
def get_content_recommendations(movie_title, n=10):
    target = movies.filter(
        pl.col("title").str.contains(movie_title, literal=True)
    )

    if target.height == 0:
        return pl.DataFrame({
            "title": ["MOVIE NOT FOUND"]
        })

    target_movie = target.row(0, named=True)
    target_genres = target_movie["genre_list"]

    results = []

    for row in movies.iter_rows(named=True):
        if row["movieId"] == target_movie["movieId"]:
            continue

        similarity = genre_similarity(
            target_genres,
            row["genre_list"]
        )

        results.append({
            "movieId": row["movieId"],
            "title": row["title"],
            "genres": row["genres"],
            "similarity": similarity
        })

    return (
        pl.DataFrame(results)
        .sort("similarity", descending=True)
        .head(n)
    )

# Get rating scores
def get_movie_rating_stats():
    return (
        ratings
        .group_by("movieId")
        .agg([
            pl.len().alias("rating_count"),
            pl.col("rating").mean().alias("average_rating")
        ])
    )
    
def get_genre_scores(user_id):
    user_ratings = ratings.filter(
        pl.col("userId") == user_id
    )

    user_movies = user_ratings.join(
        movies,
        on="movieId",
        how="left"
    )

    liked_movies = user_movies.filter(
        pl.col("rating") >= 4.0
    )

    liked_genres = liked_movies.with_columns(
        pl.col("genres")
        .str.split("|")
        .alias("genre_list")
    )

    genre_preferences = (
        liked_genres
        .explode("genre_list")
        .group_by("genre_list")
        .len()
        .sort("len", descending=True)
    )

    return dict(
        zip(
            genre_preferences["genre_list"],
            genre_preferences["len"]
        )
    )

# Compare to personal rated movies
def personalized_score(genres, genre_scores):
    if not genre_scores:
        return 0.0

    genre_list = genres.split("|")

    if not genre_list:
        return 0.0

    max_genre_score = max(genre_scores.values())

    if max_genre_score == 0:
        return 0.0

    score = sum(
        genre_scores.get(genre, 0)
        for genre in genre_list
    ) / len(genre_list)

    return score / max_genre_score


def recommend_for_user(movie_title, user_id, n=10):
    n = min(n, 10)
    
    # Get this user's genre preferences
    genre_scores = get_genre_scores(user_id)

    # Get content-based recommendations
    recommendations = get_content_recommendations(
        movie_title,
        n=10
    )
    
    # Get rated movies
    user_rated_movies = ratings.filter(
        pl.col("userId") == user_id
    )["movieId"]
    
    # Remove rated movies
    recommendations = recommendations.filter(
        ~pl.col("movieId").is_in(user_rated_movies)
    )

    # Calculate personalized score
    recommendations = recommendations.with_columns(
        pl.col("genres")
        .map_elements(
            lambda genres: personalized_score(
                genres,
                genre_scores
            ),
            return_dtype=pl.Float64
        )
        .alias("personalized_score")
    )

    # Calculate final hybrid score
    recommendations = recommendations.with_columns(
        (
            0.5 * pl.col("similarity")
            + 0.5 * pl.col("personalized_score")
        ).alias("final_score")
    )

    # Add rating statistics
    recommendations = recommendations.join(
        get_movie_rating_stats(),
        on="movieId",
        how="left"
    )
    
    # Sort by final score
    return (
        recommendations
        .sort("final_score", descending=True)
        .head(n)
    )

In [28]:
recommend_for_user("Sense and Sensibility", 109)

C:\Users\hatha\AppData\Local\Temp\ipykernel_19752\699161049.py:85: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("genre_list")
C:\Users\hatha\AppData\Local\Temp\ipykernel_19752\699161049.py:139: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  recommendations = recommendations.filter(


movieId,title,genres,similarity,personalized_score,final_score,rating_count,average_rating
i64,str,str,f64,f64,f64,u32,f64
25,"""Leaving Las Vegas (1995)""","""Drama|Romance""",1.0,0.722772,0.861386,22525,3.676182
28,"""Persuasion (1995)""","""Drama|Romance""",1.0,0.722772,0.861386,3310,4.041239
35,"""Carrington (1995)""","""Drama|Romance""",1.0,0.722772,0.861386,1497,3.485304
46,"""How to Make an American Quilt …","""Drama|Romance""",1.0,0.722772,0.861386,3179,3.267852
49,"""When Night Is Falling (1995)""","""Drama|Romance""",1.0,0.722772,0.861386,263,3.528517
74,"""Bed of Roses (1996)""","""Drama|Romance""",1.0,0.722772,0.861386,3565,3.289621
83,"""Once Upon a Time... When We We…","""Drama|Romance""",1.0,0.722772,0.861386,589,3.640917
85,"""Angels and Insects (1995)""","""Drama|Romance""",1.0,0.722772,0.861386,2703,3.523122
105,"""Bridges of Madison County, The…","""Drama|Romance""",1.0,0.722772,0.861386,10131,3.33511


In [29]:
recommend_for_user("Sense and Sensibility", 244)


C:\Users\hatha\AppData\Local\Temp\ipykernel_19752\699161049.py:85: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("genre_list")
C:\Users\hatha\AppData\Local\Temp\ipykernel_19752\699161049.py:139: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  recommendations = recommendations.filter(


movieId,title,genres,similarity,personalized_score,final_score,rating_count,average_rating
i64,str,str,f64,f64,f64,u32,f64
25,"""Leaving Las Vegas (1995)""","""Drama|Romance""",1.0,0.588889,0.794444,22525,3.676182
28,"""Persuasion (1995)""","""Drama|Romance""",1.0,0.588889,0.794444,3310,4.041239
35,"""Carrington (1995)""","""Drama|Romance""",1.0,0.588889,0.794444,1497,3.485304
46,"""How to Make an American Quilt …","""Drama|Romance""",1.0,0.588889,0.794444,3179,3.267852
49,"""When Night Is Falling (1995)""","""Drama|Romance""",1.0,0.588889,0.794444,263,3.528517
74,"""Bed of Roses (1996)""","""Drama|Romance""",1.0,0.588889,0.794444,3565,3.289621
83,"""Once Upon a Time... When We We…","""Drama|Romance""",1.0,0.588889,0.794444,589,3.640917
85,"""Angels and Insects (1995)""","""Drama|Romance""",1.0,0.588889,0.794444,2703,3.523122
105,"""Bridges of Madison County, The…","""Drama|Romance""",1.0,0.588889,0.794444,10131,3.33511
